<a href="https://colab.research.google.com/github/aray-png/Problem-Set-3-Econ_3916_PS3.ipynb/blob/main/Copy_of_ps3_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Abigail Ray || Econ 3916 PS3 — The Number Nobody Can Reproduce

**ECON 3916 · Applied Data Analytics** · Topics 9-11 · 100 points · your role: Analyst, Business Intelligence

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. Write the SQL yourself; that is the skill being assessed and it is the one interviewers test. AI is authorised and required in Phase 4.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Reproduce Both Numbers (25 pts)

### Step 1.1: Build the warehouse (8 pts)

In [2]:
!pip install -q duckdb                      # the brief's setup step 2 (quick if Colab already has it)
import duckdb
import statsmodels.formula.api as smf       # for Phase 3

con = duckdb.connect()                       # your warehouse: register customers and orders here

# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_CUSTOMERS = 3_000          # at least 3,000 customers ...
MIN_WEST = 2_000             # ... and at least 2,000 of them in the West
NB_N, NB_P = 0.8, 0.12       # n_orders = 1 + NegativeBinomial(n=0.8, p=0.12): mean near 7, long right tail
REV_PER_ORDER = 60           # revenue = 60 * n_orders + Normal(400, 150), floored at 50
REV_MEAN, REV_SD, REV_FLOOR = 400, 150, 50

In [3]:
# Step 1.1
regions = np.concatenate([
    np.full(MIN_WEST, 'West'),
    rng.choice(['East', 'North', 'South'], size=N_CUSTOMERS - MIN_WEST),
])
regions = rng.permutation(regions)
assert (regions == 'West').sum() >= MIN_WEST
n_orders = 1 + rng.negative_binomial(NB_N, NB_P, size=N_CUSTOMERS)

revenue = REV_PER_ORDER * n_orders + rng.normal(REV_MEAN, REV_SD, size=N_CUSTOMERS)
revenue = np.maximum(revenue, REV_FLOOR).round(2)
start = np.datetime64('2020-01-01')
signup_date = start + rng.integers(0, 365 * 5, size=N_CUSTOMERS).astype('timedelta64[D]')

customers = pd.DataFrame({
    'customer_id': np.arange(1, N_CUSTOMERS + 1),
    'region': regions,
    'revenue': revenue,
    'signup_date': signup_date,
    'n_orders': n_orders,          # kept for checking; the orders table is the source of truth
})

# One row in orders per order a customer placed.
order_cust = np.repeat(customers['customer_id'].to_numpy(), n_orders)
order_signup = np.repeat(signup_date, n_orders)
orders = pd.DataFrame({
    'order_id': np.arange(1, len(order_cust) + 1),
    'customer_id': order_cust,
    'order_date': order_signup + rng.integers(0, 365, size=len(order_cust)).astype('timedelta64[D]'),
})

con.register('customers', customers)
con.register('orders', orders)

# Confirm both tables are queryable and match the design.
display(con.sql("""
    SELECT region, COUNT(*) AS customers, ROUND(AVG(n_orders), 2) AS avg_orders,
           MAX(n_orders) AS max_orders, ROUND(AVG(revenue), 1) AS avg_revenue
    FROM customers GROUP BY region ORDER BY customers DESC
""").df())
display(con.sql("SELECT COUNT(*) AS order_rows, COUNT(DISTINCT customer_id) AS customers_with_orders FROM orders").df())
print("orders rows == sum(n_orders):", len(orders) == customers['n_orders'].sum())
print("corr(revenue, n_orders):", round(np.corrcoef(revenue, n_orders)[0, 1], 3))

,region,customers,avg_orders,max_orders,avg_revenue
0,West,2000,7.00,57,823.9
1,South,336,6.87,41,812.8
2,East,334,6.25,47,761.6
3,North,330,6.58,38,803.0


,order_rows,customers_with_orders
0,20570,3000


orders rows == sum(n_orders): True
corr(revenue, n_orders): 0.944


### Step 1.2: Write both queries (7 pts)

In [4]:
# Step 1.2
# low number
low_sql = """
SELECT COUNT(*)     AS rows_averaged,
       AVG(revenue) AS avg_revenue
FROM customers
WHERE region = 'West'
"""

# high number
high_sql = """
SELECT COUNT(*)       AS rows_averaged,
       AVG(c.revenue) AS avg_revenue
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
WHERE c.region = 'West'
"""

low  = con.sql(low_sql).df()
high = con.sql(high_sql).df()
display(low, high)

ratio = high.avg_revenue[0] / low.avg_revenue[0]
print(f"Low:  ${low.avg_revenue[0]:,.0f} over {low.rows_averaged[0]:,} rows")
print(f"High: ${high.avg_revenue[0]:,.0f} over {high.rows_averaged[0]:,} rows")
print(f"Ratio: {ratio:.3f}")

# Verification checkpoint from the brief
assert 780 <= low.avg_revenue[0] <= 845
assert 1160 <= high.avg_revenue[0] <= 1320
assert 1.45 <= ratio <= 1.60

,rows_averaged,avg_revenue
0,2000,823.93


,rows_averaged,avg_revenue
0,14004,1256.763


Low:  $824 over 2,000 rows
High: $1,257 over 14,004 rows
Ratio: 1.525


> **🛑 Verification checkpoint.** On West-region customers you should get about
> **$812** from the un-joined query (accept $780 to $845) and about **$1,240**
> from the joined one (accept $1,160 to $1,320), a ratio between **1.45 and
> 1.60**. If the two numbers are close together, your revenue is not correlated
> with order count, there is no fan-out to find, and the rest of the problem set
> has nothing to work on — go back to Step 1.1. **Do not proceed until it
> matches.**

*Your answer here.*
 Which number is correct?

**The correct number is $824**, the un-joined average over 2,000 West customers.

The question is "what is the average revenue of a West customer?", which makes the **customer** the unit of observation, so each customer should count exactly once. The `customers` table has one row per customer, so averaging `revenue` there answers the question directly.

The joined query silently changes the unit of observation from the customer to the order. After joining `orders`, each customer's revenue is repeated once for every order they placed, which turns 2,000 customers into 14,004 rows. A customer with 40 orders now counts 40 times while a customer with one order counts once.

That weighting would be harmless if order count had nothing to do with revenue, however it does and revenue rises with order count. So the rows the join duplicates are disproportionately the high revenue accounts, and the average is pulled up to **$1,257**.

It is key to ntoe that the join added no new revenue information it only changed how much each customer counts.

### Step 1.3: The rule (10 pts)

In [19]:
import warnings
warnings.simplefilter("always", UserWarning)                   # always show OUR fan-out warning
warnings.filterwarnings("ignore", category=DeprecationWarning)  # hide Colab's internal noise

def check_fanout(con, base_sql, query_sql, label="query"):
    """Catch a join that changes the row count before anything gets averaged.

    base_sql:  the rows you MEAN to aggregate over -- one row per unit of observation.
    query_sql: the rows the query ACTUALLY builds, joins included (its FROM/JOIN/WHERE,
               selecting rows instead of the aggregate).
    Warns and returns False if the join changed the row count; returns True otherwise.
    """
    n_base  = con.execute(f"SELECT COUNT(*) FROM ({base_sql})").fetchone()[0]
    n_query = con.execute(f"SELECT COUNT(*) FROM ({query_sql})").fetchone()[0]
    if n_query != n_base:
        warnings.warn(
            f"FAN-OUT in {label}: {n_base:,} base rows became {n_query:,} "
            f"({n_query / n_base:.2f}x). Each unit is being counted more than once.",
            stacklevel=2)
        return False
    print(f"OK {label}: {n_query:,} rows, same as base. No fan-out.")
    return True

#one row per West customer
base = "SELECT * FROM customers WHERE region = 'West'"

#customers joined to every one of their orders
bad_rows = """
SELECT c.* FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
WHERE c.region = 'West'"""

#no join
good_rows = "SELECT * FROM customers WHERE region = 'West'"

#a join done safely
safe_join_rows = """
SELECT c.*, o.order_count FROM customers c
JOIN (SELECT customer_id, COUNT(*) AS order_count FROM orders GROUP BY customer_id) o
  ON o.customer_id = c.customer_id
WHERE c.region = 'West'"""

assert check_fanout(con, base, bad_rows,       "high query ($1,257)") is False   # fires
assert check_fanout(con, base, good_rows,      "low query ($824)")    is True    # silent
assert check_fanout(con, base, safe_join_rows, "join to pre-aggregated orders") is True

OK low query ($824): 1,700 rows, same as base. No fan-out.
OK join to pre-aggregated orders: 1,700 rows, same as base. No fan-out.


/tmp/ipykernel_3812/1921206381.py:44: UserWarning: FAN-OUT in high query ($1,257): 1,700 base rows became 11,884 (6.99x). Each unit is being counted more than once.
  assert check_fanout(con, base, bad_rows,       "high query ($1,257)") is False   # fires


*Your answer here.*
Before averaging anything, it counts the rows you meant to average over and the rows the query actually builds. A join to a one-to-many table like orders multiplies rows, so the counts disagree and the check warns. The 6.99x in the warning is the average number of orders per West customer, which is exactly how much the join inflated the row count.

## Phase 2: The Rows That Disappear (30 pts)

### Step 2.1: Introduce the unknown (5 pts)

In [20]:
NULL_SHARE = 0.15

if 'region_original' not in globals():
    region_original = customers['region'].copy()

is_missing = rng.random(len(customers)) < NULL_SHARE
customers['region'] = region_original.where(~is_missing, None)   # None -> NULL in DuckDB

# Re-register so DuckDB sees the updated column.
con.unregister('customers')
con.register('customers', customers)

# Confirm the NULLs are real SQL NULLs: COUNT(region) skips them, COUNT(*) doesn't.
display(con.sql("""
    SELECT COUNT(*)                     AS customers,
           COUNT(region)                AS region_known,
           COUNT(*) - COUNT(region)     AS region_null,
           ROUND(100.0 * (COUNT(*) - COUNT(region)) / COUNT(*), 1) AS pct_null
    FROM customers
""").df())

display(con.sql("""
    SELECT region, COUNT(*) AS customers
    FROM customers GROUP BY region ORDER BY customers DESC
""").df())

,customers,region_known,region_null,pct_null
0,3000,2541,459,15.3


,region,customers
0,West,1684
1,None,459
2,South,290
3,North,287
4,East,280


### Step 2.2: Lose them, then find them (15 pts)

In [21]:
def count_rows(sql):
    return con.execute(sql).fetchall()[0][0]
#original analyst
lost = count_rows("SELECT COUNT(*) FROM customers WHERE region != 'West'")

#everyone not known to be West, unknowns included
found = count_rows("""
    SELECT COUNT(*) FROM customers
    WHERE region IS DISTINCT FROM 'West'      -- same as: region != 'West' OR region IS NULL
""")

nulls = count_rows("SELECT COUNT(*) FROM customers WHERE region IS NULL")

print(f"WHERE region != 'West':               {lost:,} rows")
print(f"WHERE region IS DISTINCT FROM 'West': {found:,} rows")
print(f"Difference: {found - lost:,} rows  (NULL-region customers: {nulls:,})")
assert found - lost == nulls

WHERE region != 'West':               857 rows
WHERE region IS DISTINCT FROM 'West': 1,316 rows
Difference: 459 rows  (NULL-region customers: 459)


*Your answer here.*

The main difference is `WHERE region != 'West'` returns 843 rows, while the query the analyst meant, `WHERE region IS DISTINCT FROM 'West'`, returns 1,316. The 459 missing rows are exactly the customers whose region is NULL.

The mechanism of `NULL != 'West'` evaluates to UNKNOWN rather than TRUE, and `WHERE` keeps only rows that are TRUE, so every customer with a missing region is silently dropped. If missing-region customers are systematically different, than the first query removes them from every regional cut without any error or warning. Every region then looks older, larger and more loyal than it really is, and comparisons between regions are made on a population that has quietly excluded the riskiest 15% of customers. Any decision built on those numbers, such as where to invest or which region is underperforming, is given that bias.

### Step 2.3: Aggregate without collapsing (10 pts)

In [10]:
window_sql = """
SELECT customer_id,
       region,
       revenue,
       AVG(revenue) OVER w                         AS region_avg_revenue,
       revenue - AVG(revenue) OVER w               AS diff_from_region_avg,
       CASE WHEN revenue >= AVG(revenue) OVER w
            THEN 'above' ELSE 'below' END          AS vs_region_avg
FROM customers
WINDOW w AS (PARTITION BY region)
ORDER BY customer_id
"""

result = con.sql(window_sql).df()
display(result.head(10))

n_before = con.execute("SELECT COUNT(*) FROM customers").fetchall()[0][0]
print(f"Rows in customers: {n_before:,}   Rows returned: {len(result):,}")
assert len(result) == n_before                    # row count unchanged

,customer_id,region,revenue,region_avg_revenue,diff_from_region_avg,vs_region_avg
0,1,None,843.53,826.509,17.021,above
1,2,West,546.37,820.773,-274.403,below
2,3,West,2233.20,820.773,1412.427,above
3,4,None,1171.16,826.509,344.651,above
4,5,West,687.04,820.773,-133.733,below
5,6,North,467.37,793.318,-325.948,below
6,7,West,691.78,820.773,-128.993,below
7,8,None,615.78,826.509,-210.729,below
8,9,West,635.62,820.773,-185.153,below
9,10,None,371.49,826.509,-455.019,below


Rows in customers: 3,000   Rows returned: 3,000


## Phase 3: The Causal Question (25 pts)

### Step 3.1: The programme evaluation (12 pts)

In [11]:
# earlier cells won't change these numbers
rng3 = np.random.default_rng(3916)

N_SIM = 10_000
TRUE_EFFECT = 25

#existing account size.
account_size = rng3.normal(50, 15, N_SIM)

# Bigger accounts = likelyhood in west so get the programme
p_west = 1 / (1 + np.exp(-(account_size - 50) / 8))
west = (rng3.random(N_SIM) < p_west).astype(int)

# Bigger accounts = more revenue
revenue = 300 + 10 * account_size + TRUE_EFFECT * west + rng3.normal(0, 100, N_SIM)

sim = pd.DataFrame({'account_size': account_size, 'west': west, 'revenue': revenue})

display(sim.groupby('west').agg(customers=('revenue', 'size'),
                                avg_account_size=('account_size', 'mean'),
                                avg_revenue=('revenue', 'mean')).round(1))

naive    = smf.ols('revenue ~ west', data=sim).fit()
adjusted = smf.ols('revenue ~ west + account_size', data=sim).fit()   #hold account size fixed

results = pd.DataFrame({
    'estimate': [TRUE_EFFECT, naive.params['west'], adjusted.params['west']],
    'ci_low':   [np.nan, naive.conf_int().loc['west', 0], adjusted.conf_int().loc['west', 0]],
    'ci_high':  [np.nan, naive.conf_int().loc['west', 1], adjusted.conf_int().loc['west', 1]],
}, index=['truth', 'naive', 'adjusted']).round(1)
results['bias'] = (results['estimate'] - TRUE_EFFECT).round(1)
display(results)

,customers,avg_account_size,avg_revenue
west,,,
0,4898,41.0,710.2
1,5102,58.9,914.9


,estimate,ci_low,ci_high,bias
truth,25.0,NaN,NaN,0.0
naive,204.7,198.5,210.9,179.7
adjusted,26.9,22.1,31.8,1.9


*Your answer here.*

The naive comparison says the programme is worth 205 dollars per customer. The true effect is $25. The naive number is about eight times too large, and its confidence interval doesn't come close to the truth, so more data wouldn't fix it. This is because the west customers already had larger accounts before the programme (58.9 vs 41), and larger accounts earn more regardless of the programme.

Holding account size fixed, comparing West and non-West customers of the same size, the estimate drops to 26.9 dollars, which contains the true $25. The programme works, but its effect is small. Most of West's revenue advantage comes from who was in the West to begin with.

### Step 3.2: The control that ruins it (13 pts)

In [22]:
ticket_rate = 1 + 3 * sim['west'] + 0.01 * sim['revenue']
sim['tickets'] = rng3.poisson(ticket_rate)

display(sim.groupby('west')['tickets'].mean().round(2).rename('avg_tickets'))

with_tickets = smf.ols('revenue ~ west + account_size + tickets', data=sim).fit()

results.loc['adjusted + tickets'] = [
    with_tickets.params['west'],
    with_tickets.conf_int().loc['west', 0],
    with_tickets.conf_int().loc['west', 1],
    with_tickets.params['west'] - TRUE_EFFECT,
]
display(results.round(1))

,avg_tickets
west,
0,8.15
1,13.16


,estimate,ci_low,ci_high,bias
truth,25.0,NaN,NaN,0.0
naive,204.7,198.5,210.9,179.7
adjusted,26.9,22.1,31.8,1.9
adjusted + tickets,-1.7,-6.6,3.3,-26.7


*Your answer here.*

Controlling for support tickets sounds sensible, but here it erases a real result. Tickets aren't a background trait like account size, rather they're something that happens after the programme starts, and they rise for two reasons. The first is because of the programme itself and the second is because of higher revenue. So when comparing West and non-West customers who filed the same number of tickets, we aren't comparing like with like. A West customer with 10 tickets got several of them from the programme, which means their revenue must be lower to land at 10. A non-West customer with 10 tickets got them all from being a busy, high-revenue account. Holding tickets fixed lines up lower-revenue West customers against higher-revenue non-West customers, and the programme's genuine $25 gain is cancelled out. This makes the estimate drop to about 3 dollars, as if the programme did nothing. The rule is to adjust for things that were true before the programme started, like account size, and never for things the programme or the outcome helped produce.

## Phase 4: AI Expansion — The Co-Pilot Era (20 pts)

**Status: AI usage AUTHORISED and REQUIRED.** Use the **P.R.I.M.E. framework**.

### Task 4.1: The query reviewer (12 pts)

In [8]:
def review_query(con, sql):
    """The model's function from the P.R.I.M.E. prompt, then your corrections:
    returns a list of structured warnings for one SELECT."""
    ...

In [13]:
## AI's output one after posting the prompt as is.
import re
import pandas as pd

# Aggregates whose answer depends on how many times each row appears.
_AGG_RE = re.compile(
    r"\b(AVG|SUM|COUNT|MEDIAN|STDDEV\w*|VAR\w*|MEAN)\s*\(\s*(DISTINCT\s+)?([^()]*)\)",
    re.IGNORECASE)
_KEYWORDS = {"WHERE", "JOIN", "ON", "USING", "GROUP", "ORDER", "LEFT", "RIGHT", "INNER",
             "FULL", "OUTER", "CROSS", "WINDOW", "HAVING", "LIMIT", "QUALIFY", "UNION", "AS"}


def _strip_subqueries(sql):
    """Replace every parenthesised (SELECT ...) with a placeholder so the top-level
    FROM/JOIN/WHERE can be read without confusing it with an inner query's."""
    out, i, subs = [], 0, []
    while i < len(sql):
        if sql[i] == "(" and re.match(r"\(\s*SELECT\b", sql[i:], re.IGNORECASE):
            depth, j = 0, i
            while j < len(sql):
                depth += sql[j] == "("
                depth -= sql[j] == ")"
                j += 1
                if depth == 0:
                    break
            subs.append(sql[i:j])
            out.append(f"__SUBQUERY_{len(subs) - 1}__")
            i = j
        else:
            out.append(sql[i])
            i += 1
    return "".join(out), subs


def _clean(sql):
    sql = re.sub(r"--[^\n]*", " ", sql)          # drop line comments
    return re.sub(r"\s+", " ", sql).strip()


def _columns(con, table):
    return set(con.execute(f"SELECT * FROM {table} LIMIT 0").df().columns)


def review_query(con, sql):
    """Review one SELECT against the live DuckDB tables (customers, orders).

    Returns a list of dicts: {check, severity, message, detail}.
    severity is 'high' (number is likely wrong), 'medium', or 'info'.
    """
    warnings = []
    flat, subqueries = _strip_subqueries(_clean(sql))

    # ---- 1. Read the top-level FROM and JOINs -------------------------------
    # The negative lookahead stops a keyword ("FROM customers JOIN ...") being
    # swallowed as an alias, which would hide the JOIN that follows it.
    kw = "|".join(_KEYWORDS)
    refs = re.findall(
        rf"\b(FROM|JOIN)\s+([A-Za-z_]\w*)(?:\s+(?:AS\s+)?(?!(?:{kw})\b)([A-Za-z_]\w*))?",
        flat, re.IGNORECASE)
    alias_to_table, joins, base_table, base_alias = {}, [], None, None
    for kind, table, alias in refs:
        alias = alias if alias and alias.upper() not in _KEYWORDS else table
        if table.startswith("__SUBQUERY_"):
            # A join to a derived table: we can't see its grain without running it.
            warnings.append(dict(
                check="unreviewed_join", severity="info",
                message=f"{kind.upper()} to a subquery ({alias}) was not checked for fan-out.",
                detail={"subquery": subqueries[int(re.search(r'\d+', table).group())]}))
            alias_to_table[alias] = None
            continue
        alias_to_table[alias] = table
        alias_to_table[table] = table
        if kind.upper() == "FROM" and base_table is None:
            base_table, base_alias = table, alias
        elif kind.upper() == "JOIN":
            joins.append((table, alias))

    if base_table is None:
        return warnings

    cols = {t: _columns(con, t) for t in set(v for v in alias_to_table.values() if v)}

    def resolve(ref):
        """'c.region' or 'region' -> (table, column), or None if unknown."""
        if "." in ref:
            a, c = ref.split(".", 1)
            t = alias_to_table.get(a)
            return (t, c) if t else None
        owners = [t for t, cs in cols.items() if ref in cs]
        return (owners[0], ref) if len(owners) == 1 else None

    # ---- 2. Fan-out: does any join change the row count? --------------------
    # HOW WE DETECT IT WITHOUT RUNNING THE FULL JOIN TWICE:
    # A join can only multiply rows if its key repeats on the joined side. So we
    # collapse the joined table to ONE row per key with its count,
    #     (SELECT key, COUNT(*) AS n FROM orders GROUP BY key)
    # and join the base table to that small summary instead. SUM(n) is exactly the
    # number of rows the real join would produce, but the fanned-out result is never
    # materialised: the summary has at most one row per key, so this join is
    # one-to-one. Comparing SUM(n) with COUNT(*) of the base table tells us whether,
    # and by how much, the real join changes the row count.
    fanned_out = False
    for table, alias in joins:
        on = re.search(
            rf"\bJOIN\s+{table}(?:\s+(?:AS\s+)?{alias})?\s+(ON\s+(.*?)|USING\s*\((\w+)\))"
            r"(?=\s+(?:LEFT|RIGHT|INNER|FULL|CROSS)?\s*JOIN\b|\s+WHERE\b|\s+GROUP\b|"
            r"\s+ORDER\b|\s+WINDOW\b|\s+HAVING\b|\s+LIMIT\b|$)",
            flat, re.IGNORECASE)
        if not on:
            continue
        if on.group(3):                                   # USING (customer_id)
            base_key = join_key = on.group(3)
        else:
            pair = re.search(r"([\w]+)\.(\w+)\s*=\s*([\w]+)\.(\w+)", on.group(2))
            if not pair:
                continue
            a1, c1, a2, c2 = pair.groups()
            if a1 == alias:
                join_key, other_alias, base_key = c1, a2, c2
            else:
                join_key, other_alias, base_key = c2, a1, c1
            if alias_to_table.get(other_alias) != base_table:
                continue                                  # chained join; not handled

        is_left = re.search(rf"\bLEFT\s+(?:OUTER\s+)?JOIN\s+{table}\b", flat, re.IGNORECASE)
        per_row = "GREATEST(COALESCE(s.n, 0), 1)" if is_left else "COALESCE(s.n, 0)"
        base_rows, joined_rows, max_n = con.execute(f"""
            SELECT COUNT(*), SUM({per_row}), MAX(s.n)
            FROM {base_table} b
            LEFT JOIN (SELECT {join_key} AS k, COUNT(*) AS n
                       FROM {table} GROUP BY {join_key}) s
              ON b.{base_key} = s.k
        """).fetchall()[0]

        if joined_rows != base_rows:
            fanned_out = joined_rows > base_rows
            warnings.append(dict(
                check="join_changes_row_count",
                severity="high" if fanned_out else "medium",
                message=(f"JOIN {table} turns {base_rows:,} {base_table} rows into "
                         f"{int(joined_rows):,} ({joined_rows / base_rows:.2f}x). "
                         + (f"{table}.{join_key} is not unique (up to {max_n} rows per key)."
                            if fanned_out else "Unmatched rows are being dropped.")),
                detail={"base_table": base_table, "joined_table": table,
                        "base_rows": int(base_rows), "joined_rows": int(joined_rows),
                        "multiplier": round(joined_rows / base_rows, 3)}))

    # ---- 3. Aggregate computed over a fanned-out result ----------------------
    # If rows were multiplied, any AVG/SUM/COUNT of a column that lives on the base
    # table counts each base row once per match. The number changes even though
    # nothing about the base table did.
    if fanned_out:
        select_list = re.search(r"\bSELECT\b(.*?)\bFROM\b", flat, re.IGNORECASE)
        for fn, distinct, arg in _AGG_RE.findall(select_list.group(1) if select_list else ""):
            arg = arg.strip()
            target = resolve(arg) if arg != "*" else (base_table, "*")
            if target and target[0] == base_table:
                warnings.append(dict(
                    check="aggregate_over_fanout",
                    severity="medium" if distinct else "high",
                    message=(f"{fn.upper()}({distinct}{arg}) is computed after a fan-out, so "
                             f"each {base_table} row is weighted by its number of matches."
                             + (" DISTINCT dedups values, not rows: two customers with the "
                                "same revenue collapse to one." if distinct else "")),
                    detail={"function": fn.upper(), "argument": arg, "distinct": bool(distinct)}))

    # ---- 4. Filters on nullable columns --------------------------------------
    # WHY THIS IS A CORRECTNESS ISSUE, NOT STYLE:
    # SQL uses three-valued logic: a comparison is TRUE, FALSE, or UNKNOWN. Any
    # comparison with NULL (NULL != 'West', NULL > 100) is UNKNOWN, and WHERE keeps
    # only TRUE rows. So `region != 'West'`, which reads as "everyone except West",
    # actually means "everyone whose region is known and isn't West". The rows with
    # unknown region vanish with no error, and the result answers a different
    # question than the one asked. That changes the number, not just its readability.
    where = re.search(
        r"\bWHERE\b(.*?)(?=\bGROUP\s+BY\b|\bORDER\s+BY\b|\bWINDOW\b|\bHAVING\b|"
        r"\bLIMIT\b|\bQUALIFY\b|$)", flat, re.IGNORECASE)
    if where:
        clause = where.group(1)
        preds = re.findall(
            r"([A-Za-z_][\w]*(?:\.\w+)?)\s*(!=|<>|<=|>=|=|<|>|\s+NOT\s+IN\b|\s+NOT\s+LIKE\b|\s+IN\b|\s+LIKE\b)",
            clause, re.IGNORECASE)
        seen = set()
        for ref, op in preds:
            op = " ".join(op.upper().split())
            target = resolve(ref)
            if not target or (ref, op) in seen:
                continue
            seen.add((ref, op))
            table, col = target
            if re.search(rf"\b{re.escape(ref)}\s+IS\s+NULL\b", clause, re.IGNORECASE):
                continue                                   # unknowns explicitly handled
            n_null = con.execute(
                f"SELECT COUNT(*) - COUNT({col}) FROM {table}").fetchall()[0][0]
            if n_null == 0:
                continue
            negation = op in {"!=", "<>", "NOT IN", "NOT LIKE"}
            warnings.append(dict(
                check="null_filter_drops_unknowns",
                severity="high" if negation else "info",
                message=(f"WHERE {ref} {op} ... silently drops the {n_null:,} rows where "
                         f"{table}.{col} IS NULL."
                         + (" A negated filter reads as 'everything except', so these rows "
                            "are almost certainly meant to be kept. Use IS DISTINCT FROM "
                            "or add OR ... IS NULL." if negation else
                            " Rows with an unknown value can't match, even if their true "
                            "value would have. Confirm that's intended.")),
                detail={"table": table, "column": col, "operator": op,
                        "null_rows_dropped": int(n_null)}))

    return warnings


def as_frame(warnings):
    """Convenience: show the structured warnings as a table."""
    return pd.DataFrame(warnings, columns=["check", "severity", "message", "detail"])

In [15]:
# ---- Revision: check grain, not just row counts ----
GRAIN = {'customers': 'customer_id', 'orders': 'order_id'}   # one row per ... in each table

def check_grain(con, sql, grain=GRAIN):
    """For every aggregate, check that the table its column comes from is still
    one-row-per-unit in the rows the query actually aggregates (after joins AND WHERE)."""
    warnings = []
    flat, subqueries = _strip_subqueries(_clean(sql))
    body = re.search(r"\bFROM\b(.*?)(?=\bGROUP\s+BY\b|\bORDER\s+BY\b|\bWINDOW\b|\bHAVING\b|\bLIMIT\b|$)",
                     flat, re.IGNORECASE)
    select_list = re.search(r"\bSELECT\b(.*?)\bFROM\b", flat, re.IGNORECASE)
    if not body or not select_list:
        return warnings
    from_where = body.group(1)
    for i, sub in enumerate(subqueries):                 # put subqueries back so the count is real
        from_where = from_where.replace(f"__SUBQUERY_{i}__", sub)

    kw = "|".join(_KEYWORDS)
    aliases = {}
    for _, table, alias in re.findall(
            rf"\b(FROM|JOIN)\s+([A-Za-z_]\w*)(?:\s+(?:AS\s+)?(?!(?:{kw})\b)([A-Za-z_]\w*))?",
            flat, re.IGNORECASE):
        aliases[alias or table] = table
        aliases[table] = table
    tables = set(aliases.values())

    for fn, distinct, arg in _AGG_RE.findall(select_list.group(1)):
        arg = arg.strip()
        if arg == "*":
            continue
        if "." in arg:
            a, col = arg.split(".", 1)
            table = aliases.get(a)
        else:
            a, col = None, arg
            owners = [t for t in tables if t in grain and
                      col in set(con.execute(f"SELECT * FROM {t} LIMIT 0").df().columns)]
            table = owners[0] if len(owners) == 1 else None
        if table not in grain:
            continue
        key = f"{a}.{grain[table]}" if a else grain[table]
        # ONE pass over the joined, filtered rows: are there more rows than units?
        rows, units = con.execute(
            f"SELECT COUNT(*), COUNT(DISTINCT {key}) FROM {from_where}").fetchall()[0]
        if rows != units:
            warnings.append(dict(
                check="aggregate_at_wrong_grain", severity="high",
                message=(f"{fn.upper()}({arg}) aggregates a {table}-level column over {rows:,} rows "
                         f"but only {units:,} distinct {table} ({rows / units:.2f}x). "
                         f"Each {table} row is counted once per match."),
                detail={"table": table, "rows": int(rows), "units": int(units),
                        "multiplier": round(rows / units, 3)}))
    return warnings

def review_query_v2(con, sql):
    return review_query(con, sql) + check_grain(con, sql)

In [16]:
pd.set_option('display.max_colwidth', None)

lost_sql  = "SELECT COUNT(*) FROM customers WHERE region != 'West'"
found_sql = "SELECT COUNT(*) FROM customers WHERE region IS DISTINCT FROM 'West'"

# The Step 1.3 "safe" join, written as an aggregate
safe_join_sql = """
SELECT AVG(c.revenue) FROM customers c
JOIN (SELECT customer_id, COUNT(*) AS order_count FROM orders GROUP BY customer_id) o
  ON o.customer_id = c.customer_id
WHERE c.region = 'West'"""

# The $1,240 mistake written the other way round: start from orders, join customers
flipped_sql = """
SELECT AVG(c.revenue) FROM orders o
JOIN customers c ON c.customer_id = o.customer_id"""

queries = {
    "1.2 low ($824)":     low_sql,
    "1.2 high ($1,257)":  high_sql,
    "1.3 safe join":      safe_join_sql,
    "2.2 != 'West'":      lost_sql,
    "2.2 IS DISTINCT":    found_sql,
    "2.3 window":         window_sql,
    "flipped high query": flipped_sql,
}

review = pd.concat(
    [as_frame(review_query(con, sql)).assign(query=name) for name, sql in queries.items()],
    ignore_index=True)
display(review[['query', 'severity', 'check', 'message']])

# Testable: the bad query must raise a high-severity fan-out warning; the good one must not.
assert any(w['check'] == 'aggregate_over_fanout' for w in review_query(con, high_sql))
assert not any(w['severity'] == 'high' for w in review_query(con, low_sql))

,query,severity,check,message
0,1.2 low ($824),info,null_filter_drops_unknowns,"WHERE region = ... silently drops the 457 rows where customers.region IS NULL. Rows with an unknown value can't match, even if their true value would have. Confirm that's intended."
1,"1.2 high ($1,257)",high,join_changes_row_count,"JOIN orders turns 3,000 customers rows into 20,570 (6.86x). orders.customer_id is not unique (up to 57 rows per key)."
2,"1.2 high ($1,257)",high,aggregate_over_fanout,"COUNT(*) is computed after a fan-out, so each customers row is weighted by its number of matches."
3,"1.2 high ($1,257)",high,aggregate_over_fanout,"AVG(c.revenue) is computed after a fan-out, so each customers row is weighted by its number of matches."
4,"1.2 high ($1,257)",info,null_filter_drops_unknowns,"WHERE c.region = ... silently drops the 457 rows where customers.region IS NULL. Rows with an unknown value can't match, even if their true value would have. Confirm that's intended."
5,1.3 safe join,info,unreviewed_join,JOIN to a subquery (o) was not checked for fan-out.
6,1.3 safe join,info,null_filter_drops_unknowns,"WHERE c.region = ... silently drops the 457 rows where customers.region IS NULL. Rows with an unknown value can't match, even if their true value would have. Confirm that's intended."
7,2.2 != 'West',high,null_filter_drops_unknowns,"WHERE region != ... silently drops the 457 rows where customers.region IS NULL. A negated filter reads as 'everything except', so these rows are almost certainly meant to be kept. Use IS DISTINCT FROM or add OR ... IS NULL."


In [9]:
# Run review_query on your Phase 1 and Phase 2 queries.

*Your answer here.*

Evaluate if it would have caught the $1,240?

Yes because the high query gets three high-severity warnings whereas the low query gets no high-severity warnings, only an info note.

In the Phase 1 fan-out, it did catch which aggregates it corrupts. Pluys in the Phase 2 != 'West' filter, it flagged high with the exact count of rows lost (457) and the fix. Correct silence on the NULL-safe IS DISTINCT FROM query and the window query.

I will say that it missed the same mistake with the tables swapped. FROM orders JOIN customers produces the inflated $1,243 and gets no warning. The tool measures fan-out relative to the FROM table. orders → customers is many-to-one, so the row count never changes and the damage comes from averaging a customer-level column at order grain. Catching this needs to know each table’s grain, which row counts alone can’t tell you. Row counts are measured before the WHERE.

The region = 'West' note is noise in most cases. It fires on both Phase 1 queries. It’s technically true, since unknown-region customers who are really West get excluded. That’s why it’s info, not high.
It depends on today’s data, not the schema. If no customer currently had two orders, or no region were NULL yet, it would pass queries that will break later.


### Task 4.2: The one-page writeup (8 pts)

*Your answer here.*

To: Director

Title: Analysis of west revenue and which number is right, and does the programme work?

The right number is 824 dollars, the average revenue of a West customer. The $1,257 came from the same customers after joining the orders table, which repeats each customer once per order. This means that 2,000 customers became 14,004 rows. Our biggest accounts also place the most orders, so the repeats pushed the average up about 50%. The query quietly switched from averaging customers to averaging orders.

The programme works, but its effect is small. West customers earn about 205 dollars more, but they had much larger accounts before it started. Comparing accounts of similar size, the programme adds about 27 dollars, close to the 25 dollars built into our test data. Controlling for support tickets made the answer worse (−$3) and both the programme and higher revenue generate tickets, so holding tickets fixed compares weaker West accounts with stronger ones elsewhere.

Before a number goes on a slide, I recommend:
1. State the unit. Each metric names what one row is (customer or order). The new query reviewer flags averages taken after a join has duplicated customers so it would have stopped the $1,257.
2. Make sure to count the unknowns. 457 customers have no region, and "not West" filters silently drop them. Use NULL-safe filters and report what was excluded.
3. I would adjust only for what existed before the programme, never its consequences.

### AI Transcript

Paste **your prompt exactly as you sent it** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

Model: Claude Opus 5.5

Date: November 8, 2026

Prompt:

Prep Act as an analytics engineer who reviews SQL before it reaches
a dashboard.

[Request] Write a Python function review_query(con, sql) that runs
against DuckDB and returns a list of warnings for a given SELECT. It
should flag at minimum: a join that changes the row count, a filter on
a nullable column that will silently drop unknowns, and an aggregate
computed over a fanned-out result.

[Iterate] duckdb and pandas only. Reuse my table names, customers and
orders. Return structured warnings, not printed text, so I can test it.

[Mechanism Check] Add comments explaining how you detect fan-out
without running the full join twice, and why three-valued logic makes
the null filter a correctness issue rather than a style issue.

[Evaluate] Run it against both of my Phase 1 queries and tell me
whether it would have caught the $1,240 before it reached the board.

Reply: in cell 13.

Changes: my reply to version one "can you please check grain instead of row counts. For every aggregate are the rows actually being averaged, is there still exactly one row per customer?"

Reply two: in cell 15.

* once I asked ai for it to generate based on grain rather than row counts the code in cell 15 was provided by AI. I made no manual changes given i already saw what the ai correct in its version 2, i wanted to be transparent about the work that i completed and id by lying if i said i made the adjustments without the support of AI for version 2 in cell 15.

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; your prompt is in it as sent.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 3/Econ_3916_PS3.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.